This notebook is used to verify the hypotheses infer from EDA process, by implementing basic baseline models, train differently and see the differences 

## 0. Basic pre-processing

In [1]:
import pandas as pd
from ml_challenge.config import RAW_DATA_DIR

train = pd.read_csv(RAW_DATA_DIR / "train.csv")
#train.head(5)

In [2]:
TARGET = "is_fraud"

X = train.drop(columns=["id", TARGET])
y = train[TARGET]

print(X.shape)
print(y.shape)

(699960, 24)
(699960,)


In [3]:
# split train data into train/validation
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [4]:
# Missing values? Alredy checked in EDA. There is none
# X_train.isna().sum().sort_values(ascending=True) # proof

In [5]:
# Duplicated rows (after removing id)
#X_train.groupby(X_train.columns.tolist(), as_index=False).size()
# 559968 = 699960 * 80% output rows, so no dups

In [6]:
# Scale numerical data using Standard method
# One-hot Encoding categorical data, except binary

In [7]:
num_cols = X_train.select_dtypes(["int64", "float64"]).columns.tolist()
cat_cols = X_train.select_dtypes(["str"]).columns.tolist()
bool_cols = X_train.select_dtypes(["bool"]).columns.tolist()

# print(f"Numerical columns: ({len(num_cols)}): {num_cols}")
# print(f"flagegorical columns: ({len(cat_cols)}): {cat_cols}")
# print(f"Binary columns: ({len(bool_cols)}): {bool_cols}")

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# DON'T need Imputer, data has no NaN values
num_pipeline = Pipeline([
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        drop="if_binary" # prevents creating dummy column, if category has 2 distinct values
    ))
])

bool_pipeline = Pipeline([
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        drop="if_binary"
    ))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols),
    ("bool", bool_pipeline, bool_cols)
])

## 1. Logistic Regression

In [9]:
# exp-001
# Baseline model 1
baseline_lr = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

# Training
baseline_lr.fit(X_train, y_train)

# Probality predictions
y_valid_pred = baseline_lr.predict_proba(X_valid)[:,1] # take the predicted probality of fraud

# Evaluation
from sklearn.metrics import roc_auc_score
valid_auc = roc_auc_score(y_valid, y_valid_pred)

print(f"Validation ROC-AUC: {valid_auc:.6f}")

Validation ROC-AUC: 0.940264


## 2. HistGradientBoosting baseline

In [10]:
# exp-002
# Baseline model 2
from sklearn.ensemble import HistGradientBoostingClassifier

baseline_hgb = Pipeline([
    ("preprocessor", preprocessor),
    ("model", HistGradientBoostingClassifier(
        max_iter=100,
        learning_rate=0.1,
        max_leaf_nodes=15,
        random_state=42,
    )),
])

baseline_hgb.fit(X_train, y_train)

y_valid_pred_hgb = baseline_hgb.predict_proba(X_valid)[:, 1]

hgb_auc = roc_auc_score(y_valid, y_valid_pred_hgb)

print(f"Validation ROC-AUC: {hgb_auc:.6f}")

Validation ROC-AUC: 0.942995


In [11]:
# indeed non-linear relationship, to some extent, has importance

## 3. remove Risk and velocity-related features and compare

In [16]:
# exp-003
# model: Logistic Regression
# random state: 42
# exp-003A: keep all features
# exp-003B: remove risk, velocity
# metrics: ROC-AUC

# 003A
auc_all = valid_auc

# 003B
removed_cols = [
    "merchant_risk_score",
    "velocity_score",
    "cvv_retry_count",
]

X_train_ablated = X_train.drop(columns=removed_cols)
X_valid_ablated = X_valid.drop(columns=removed_cols)

## rebuild pipeline
num_cols_ablated = X_train_ablated.select_dtypes(["int64", "float64"]).columns.tolist()
cat_cols_ablated = cat_cols # remain the same

preprocessor_ablated = ColumnTransformer([
    ("num", Pipeline([
        ("scaler", StandardScaler())
    ]), num_cols_ablated),

    ("cat", Pipeline([
        ("encode", OneHotEncoder(handle_unknown="ignore", drop="if_binary"))
    ]), cat_cols_ablated),

    ("bool", bool_pipeline, bool_cols)
])

lr_ablation = Pipeline([
    ("preprocessor", preprocessor_ablated),
    ("model", LogisticRegression(max_iter=1000)),
])

lr_ablation.fit(X_train_ablated, y_train)
y_pred_ablation = lr_ablation.predict_proba(X_valid_ablated)[:, 1]
auc_ablated = roc_auc_score(y_valid, y_pred_ablation)

print("AUC all features:", auc_all)
print("AUC without risk/velocity:", auc_ablated)
print("Difference:", auc_all - auc_ablated)

AUC all features: 0.9402638001034146
AUC without risk/velocity: 0.8827017543135731
Difference: 0.057562045789841476


This somewhat indicates the importance of those features

## 4. remove categorical features and compare

In [17]:
# exp-004
# model: Logistic Regression
# random state: 42
# exp-004A: keep all features
# exp-004B: remove categorical
# metrics: ROC-AUC

# 004A
auc_all = valid_auc

# 004B
removed_cat_cols = cat_cols

X_train_no_cat = X_train.drop(columns=removed_cat_cols)
X_valid_no_cat = X_valid.drop(columns=removed_cat_cols)

## rebuild pipeline
preprocessor_no_cat = ColumnTransformer([
    ("num", Pipeline([
        ("scaler", StandardScaler())
    ]), num_cols),

    ("bool", bool_pipeline, bool_cols)
])

lr_no_cat = Pipeline([
    ("preprocessor", preprocessor_no_cat),
    ("model", LogisticRegression(max_iter=1000)),
])

lr_no_cat.fit(X_train_no_cat, y_train)
y_pred_no_cat = lr_no_cat.predict_proba(X_valid_no_cat)[:, 1]
auc_no_cat = roc_auc_score(y_valid, y_pred_no_cat)

print("AUC all features:", auc_all)
print("AUC without categorical", auc_no_cat)
print("Difference:", auc_all - auc_no_cat)

AUC all features: 0.9402638001034146
AUC without categorical 0.9261354633304515
Difference: 0.014128336772963102


## 5. remove flags and compare

In [18]:
# exp-005
# model: Logistic Regression
# random state: 42
# exp-005A: keep all features
# exp-005B: remove Boolean flags
# metrics: ROC-AUC

# 005A
auc_all = valid_auc

# 005B
removed_cols = bool_cols

X_train_no_flag = X_train.drop(columns=removed_cols)
X_valid_no_flag = X_valid.drop(columns=removed_cols)

## rebuild pipeline
preprocessor_no_flag = ColumnTransformer([
    ("num", Pipeline([
        ("scaler", StandardScaler())
    ]), num_cols),

    ("cat", Pipeline([
        ("encode", OneHotEncoder(handle_unknown="ignore", drop="if_binary"))
    ]), cat_cols)
])

lr_no_flag = Pipeline([
    ("preprocessor", preprocessor_no_flag),
    ("model", LogisticRegression(max_iter=1000)),
])

lr_no_flag.fit(X_train_no_flag, y_train)
y_pred_no_flag = lr_no_flag.predict_proba(X_valid_no_flag)[:, 1]
auc_no_flag = roc_auc_score(y_valid, y_pred_no_flag)

print("AUC all features:", auc_all)
print("AUC without flags:", auc_no_flag)
print("Difference:", auc_all - auc_no_flag)

AUC all features: 0.9402638001034146
AUC without flags: 0.9002285680181286
Difference: 0.04003523208528603


## 6. remove money features and compare

In [20]:
# exp-006
# model: Logistic Regression
# random state: 42
# exp-006A: keep all features
# exp-006B: remove amount_usd and account_balance_usd
# metrics: ROC-AUC

# 006A
auc_all = valid_auc

# 006B
removed_cols = [
    "amount_usd",
    "account_balance_usd"
]

X_train_no_mon = X_train.drop(columns=removed_cols)
X_valid_no_mon = X_valid.drop(columns=removed_cols)

num_cols_no_mon = X_train_no_mon.select_dtypes(["int64", "float64"]).columns.tolist()

## rebuild pipeline
preprocessor_no_mon = ColumnTransformer([
    ("num", Pipeline([
        ("scaler", StandardScaler())
    ]), num_cols_no_mon),

    ("cat", Pipeline([
        ("encode", OneHotEncoder(handle_unknown="ignore", drop="if_binary"))
    ]), cat_cols),

    ("bool", bool_pipeline, bool_cols)
])

lr_no_mon = Pipeline([
    ("preprocessor", preprocessor_no_mon),
    ("model", LogisticRegression(max_iter=1000)),
])

lr_no_mon.fit(X_train_no_mon, y_train)
y_pred_no_mon = lr_no_mon.predict_proba(X_valid_no_mon)[:, 1]
auc_no_mon = roc_auc_score(y_valid, y_pred_no_mon)

print("AUC all features:", auc_all)
print("AUC without money:", auc_no_mon)
print("Difference:", auc_all - auc_no_mon)

AUC all features: 0.9402638001034146
AUC without money: 0.9402909351216352
Difference: -2.7135018220603158e-05


Requires more extensive experiment to decide if these features matters